In [5]:
import pandas as pd

region_mapping = {
    'Riyadh': 'Riyadh', 'Makkah': 'Makkah', 'Madinah': 'Madinah',
    'Qassim': 'Qassim', 'Eastern': 'Eastern Region', 'Asir': 'Aseer',
    'Tabuk': 'Tabuk', 'Hail': 'Hail', 'Northern Borders': 'Northern Borders',
    'Jazan': 'Jazan', 'Najran': 'Najran', 'Al Baha': 'Al-Baha', 'Al Jouf': 'Al-Jouf'
}

excel_file = 'Household_Energy_Statistics_2024.xlsx'

def read_sheet(sheet_name, cols):
    df = pd.read_excel(excel_file, sheet_name=sheet_name, skiprows=4)
    df = df.dropna(subset=[df.columns[1]]).iloc[:, cols]
    df.iloc[:, 0] = df.iloc[:, 0].astype(str).str.strip()
    for i in range(1, len(cols)):
        df.iloc[:, i] = pd.to_numeric(df.iloc[:, i], errors='coerce').fillna(0)
    return df

# Read ALL AC types consumption (Window, Split, Central)
ac_df = read_sheet('1-16', [1, 2, 3, 4, 5, 6, 7])
ac_df.columns = ['Region', 'WindowAC_Winter', 'WindowAC_Rest', 'SplitAC_Winter', 'SplitAC_Rest', 'CentralAC_Winter', 'CentralAC_Rest']

heater_df = read_sheet('1-14', [1, 2, 3])
heater_df.columns = ['Region', 'Heater_Winter', 'Heater_Rest']

water_heater_df = read_sheet('1-15', [1, 2, 3])
water_heater_df.columns = ['Region', 'WH_Winter', 'WH_Rest']

# Read lighting consumption
lighting_df = read_sheet('1-17', [1, 2, 3, 4, 5])
lighting_df.columns = ['Region', 'RegLamp_Winter', 'RegLamp_Rest', 'LEDLamp_Winter', 'LEDLamp_Rest']

# Read food preservation (fridge/freezer) consumption
fridge_df = read_sheet('1-19', [1, 2, 3, 4, 5])
fridge_df.columns = ['Region', 'Fridge_Winter', 'Fridge_Rest', 'Freezer_Winter', 'Freezer_Rest']

# Read thermal insulation
insulation_df = read_sheet('1-20', [1, 2])
insulation_df.columns = ['Region', 'Insulation_Yes_Pct']
insulation_df['Insulation_Yes_Pct'] = (insulation_df['Insulation_Yes_Pct'] * 100)

# Read cooking sources divided by dwelling type
dwelling_sheets = {
    'Villa': '2-9',
    'Floor in Villa': '2-10',
    'Traditional House': '2-11',
    'Floor in Traditional': '2-12',
    'Apartment': '2-13'
}

cooking_data = []
for dwelling_type, sheet in dwelling_sheets.items():
    df = read_sheet(sheet, [1, 2, 3])
    df.columns = ['Region', 'Cooking_Gas_Pct', 'Cooking_Elec_Pct']
    df['Dwelling_Type'] = dwelling_type
    cooking_data.append(df)

cooking_df = pd.concat(cooking_data, ignore_index=True)
cooking_df['Cooking_Gas_Pct'] = (cooking_df['Cooking_Gas_Pct'] * 100)
cooking_df['Cooking_Elec_Pct'] = (cooking_df['Cooking_Elec_Pct'] * 100)

# Build the dataset (Region x Season x Dwelling Type)
csv_rows = []

for reg in region_mapping.values():
    ac_row = ac_df[ac_df['Region'] == reg].iloc[0]
    heat_row = heater_df[heater_df['Region'] == reg].iloc[0]
    wh_row = water_heater_df[water_heater_df['Region'] == reg].iloc[0]

    light_row = lighting_df[lighting_df['Region'] == reg].iloc[0]
    fridge_row = fridge_df[fridge_df['Region'] == reg].iloc[0]

    insul_row = insulation_df[insulation_df['Region'] == reg].iloc[0]

    for season in ['Winter', 'Rest_of_Year']:
        for dwelling in dwelling_sheets.keys():
            cook_row = cooking_df[(cooking_df['Region'] == reg) & (cooking_df['Dwelling_Type'] == dwelling)].iloc[0]

            csv_rows.append({
                'Region': reg,
                'Season': season,
                'Dwelling_Type': dwelling,

                'Window_AC_hours_per_week': round(ac_row['WindowAC_Winter' if season == 'Winter' else 'WindowAC_Rest'], 1),
                'Split_AC_hours_per_week': round(ac_row['SplitAC_Winter' if season == 'Winter' else 'SplitAC_Rest'], 1),
                'Central_AC_hours_per_week': round(ac_row['CentralAC_Winter' if season == 'Winter' else 'CentralAC_Rest'], 1),
                'Radiant_Heater_hours_per_week': round(heat_row['Heater_Winter' if season == 'Winter' else 'Heater_Rest'], 1),
                'Water_Heater_hours_per_week': round(wh_row['WH_Winter' if season == 'Winter' else 'WH_Rest'], 1),
                'Regular_Lamps_hours_per_week': round(light_row['RegLamp_Winter' if season == 'Winter' else 'RegLamp_Rest'], 1),
                'Energy_Saving_Lamps_hours_per_week': round(light_row['LEDLamp_Winter' if season == 'Winter' else 'LEDLamp_Rest'], 1),
                'Refrigerator_hours_per_week': round(fridge_row['Fridge_Winter' if season == 'Winter' else 'Fridge_Rest'], 1),
                'Freezer_hours_per_week': round(fridge_row['Freezer_Winter' if season == 'Winter' else 'Freezer_Rest'], 1),

                'Insulation_Yes_Pct': round(insul_row['Insulation_Yes_Pct'], 1),
                'Cooking_Gas_Pct': round(cook_row['Cooking_Gas_Pct'], 1),
                'Cooking_Elec_Pct': round(cook_row['Cooking_Elec_Pct'], 1)
            })

final_df = pd.DataFrame(csv_rows)
final_df.rename(columns={'Dwelling_Type': 'Housing_type'}, inplace=True)

final_df = final_df.round(1)

final_df.to_csv('household_consumption_baseline.csv', index=False, encoding='utf-8-sig')

print("Success. Saved to household_consumption_baseline.csv")

Success. Saved to household_consumption_baseline.csv
